In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
from scipy.special import loggamma

plt.rcParams.update({
    "text.usetex":        False,
    "font.family":        "serif",
    "font.serif":         ["Palatino", "Palatino Linotype", "Georgia",
                           "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset":   "cm",
    "axes.labelsize":     11,
    "axes.titlesize":     11,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "figure.dpi":         150,
})

TROPICAL_RAINFOREST = (0.00, 0.46, 0.37)
WEBBLUE             = (0.00, 0.25, 0.60)
WEBRED              = (0.60, 0.00, 0.00)

def _pale(rgb, mix):
    return tuple(c + (1 - c) * mix for c in rgb)

def make_cmap(color, name):
    return mcolors.LinearSegmentedColormap.from_list(
        name, [_pale(color, 0.93), _pale(color, 0.35)])

def erw_mean(n_steps, p, q):
    alpha = 2 * p - 1
    beta  = 2 * q - 1
    ns    = np.arange(1, n_steps + 1, dtype=float)
    log_ratio = (loggamma(ns + alpha)
                 - loggamma(alpha + 1)
                 - loggamma(ns))
    return np.concatenate([[0.0], beta * np.exp(log_ratio)])

def simulate_erw(n, p, q=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    xi = np.empty(n + 1, dtype=np.int8)
    xi[0] = 0
    xi[1] = 1 if rng.random() < q else -1
    for t in range(1, n):
        past = xi[rng.integers(0, t) + 1]
        xi[t + 1] = past if rng.random() < p else -past
    S = np.empty(n + 1, dtype=np.int64)
    S[0] = 0
    S[1:] = np.cumsum(xi[1:])
    return S

def gradient_path(ax, x, y, cmap, vmax, lw=0.40, a=0.62):
    pts  = np.array([x, y], dtype=float).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    vals = (np.abs(y[:-1]) + np.abs(y[1:])) / 2.0
    lc   = LineCollection(segs, cmap=cmap,
                          norm=mcolors.Normalize(vmin=0, vmax=vmax),
                          linewidth=lw, alpha=a, rasterized=True)
    lc.set_array(vals)
    ax.add_collection(lc)

N_STEPS = 10000
Q       = 0.5

REGIMES = [
    (0.30, "diffusive",      r"$p = 0.30$",  TROPICAL_RAINFOREST, 8,  0.22, 0.45),
    (0.75, "critical",       r"$p = 0.75$",  WEBBLUE,             15, 0.28, 0.55),
    (0.85, "superdiffusive", r"$p = 0.85$",  WEBRED,              15, 0.28, 0.55),
]

steps = np.arange(N_STEPS + 1, dtype=float)
denom = np.where(steps == 0, 1.0, steps)

rng = np.random.default_rng()

all_paths = {
    p_val: [simulate_erw(N_STEPS, p=p_val, q=Q, rng=rng)
            for _ in range(n_paths)]
    for p_val, _, _, _, n_paths, _, _ in REGIMES
}

fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.75), sharey=False)
fig.subplots_adjust(left=0.11, right=0.98, bottom=0.20, top=0.80, wspace=0.50)

REGIME_LABEL_GREY = "#bbbbbb"

for ax, (p_val, regime, label, color, n_paths, lw, alpha) in zip(axes, REGIMES):
    cmap  = make_cmap(color, regime)
    paths = all_paths[p_val]

    normed = np.array([S / denom for S in paths])

    vmax = np.percentile(np.abs(normed[:, 1:]), 97)

    for y in normed:
        gradient_path(ax, steps, y, cmap=cmap, vmax=vmax, lw=lw, a=alpha)

    emp = np.mean(normed, axis=0)
    ax.plot(steps, emp, color=color, lw=0.65, zorder=6, solid_capstyle="round")

    ax.axhline(0, color="black", lw=0.45, ls=(0, (4, 3)), alpha=0.28, zorder=5)

    ymax = max(np.percentile(np.abs(normed[:, 1:]), 99), 0.02)
    ax.set_xlim(0, N_STEPS)
    ax.set_ylim(-ymax * 1.25, ymax * 1.25)

    ax.set_title(label, fontsize=10, pad=5, color=REGIME_LABEL_GREY)

    ax.text(0.5, 1.12, regime, transform=ax.transAxes,
            ha="center", va="bottom", fontsize=9,
            color=REGIME_LABEL_GREY, style="italic", fontfamily="serif")

    ax.set_xlabel(r"$n$", labelpad=3)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.55)
    ax.spines["bottom"].set_linewidth(0.55)
    ax.tick_params(width=0.55, length=3, direction="out")

    ax.xaxis.set_major_locator(ticker.FixedLocator([0, N_STEPS // 2, N_STEPS]))
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(
        lambda x, _: r"$0$" if x == 0
                      else (r"$5000$" if x == N_STEPS // 2 else r"$10000$")))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4, symmetric=True))

    if ax is axes[0]:
        ax.set_ylabel(r"$Z_n$", labelpad=4)

print(f"\nEmpirical mean of S_n / n at n = {N_STEPS:,}:")
for p_val, regime, label, color, n_paths, lw, alpha in REGIMES:
    normed = np.array([S / denom for S in all_paths[p_val]])
    print(f"  {regime:<15} (p = {p_val}):  {np.mean(normed[:, -1]):.6f}")

plt.savefig("figure_4_2.pdf", bbox_inches="tight")
print("Saved figure_4_2.pdf")